# Phase 3: entity resolution
Goal: find records that describe the same company under different IDs.

- Matching unit: source record (`entity_id`).
- `canonical_id` (OpenSanctions' grouping): answer key for Phase 4 only. Never used to match.
- Snapshot pinned to 20261030005356-xkx (same as Phase 2) for a clean before/after.

Steps: records, clean names, ID matches, blocking, scoring, combine, grouping, output.

### Step 1: records table
Notebook 13's pivot, grouped by `entity_id` instead of `canonical_id`. One row per source record, before any merging.
- Schemas: Company, Organization, LegalEntity, PublicBody. People excluded.
- Names: name, alias, previousName. weakAlias excluded (unreliable).
- IDs: INN, OGRN, registration number, tax number, LEI.

In [0]:
-- The query same as 13_legal_entity's notebook source but rather than grouping by canonical_ids(which means those were already identified similar entities and processed by opensanctions), we group by entity_ids so we can process everything from ground.

--Why legal entity because we only wanna include records and its linked details which are not a person that means it could either one of ('Company','Organization','LegalEntity','PublicBody')

CREATE OR REPLACE TABLE pep_dev.eval.er_records 
AS
SELECT  
    psr.entity_id,
    -- canonical_id = OpenSanctions' grouping. Answer key for Phase 4 only; never used to match.
    MAX(psr.canonical_id) AS canonical_id,
  
    collect_set(value)  FILTER( WHERE prop IN ('name', 'alias', 'previousName')) AS names,
    
    coalesce (
        MAX(CASE WHEN prop = 'jurisdiction' THEN value END),
        MAX(CASE WHEN prop = 'mainCountry' THEN value END),
        MAX(CASE WHEN prop = 'country' THEN value END)
    ) AS country,

    collect_set(value) FILTER(WHERE prop = 'innCode') AS inn_codes ,
    collect_set(value) FILTER(WHERE prop = 'ogrnCode') AS ogrn_codes,
    collect_set(value) FILTER(WHERE prop = 'registrationNumber') AS registration_numbers,
    collect_set(value) FILTER(WHERE prop = 'taxNumber') AS tax_numbers,
    collect_set(value) FILTER(WHERE prop = 'leiCode') AS lei_codes,
   
    collect_set(dataset) AS datasets,
    MAX(_snapshot_version) AS _last_seen_snapshot
    
FROM 
pep_dev.bronze.statements AS pbs
JOIN pep_dev.silver.referent AS psr
ON psr.entity_id = pbs.entity_id  
WHERE SCHEMA IN ('Company','Organization','LegalEntity','PublicBody')
AND psr.canonical_id NOT IN (SELECT canonical_id FROM pep_dev.silver.referent WHERE referent_kind = 'Person')
AND _snapshot_version='20261030005356-xkx'
-- PINNED to the Phase 2 snapshot for a clean before/after match in development phase. Gonna Switch back to the subquery below in Phase 6.

-- (
-- SELECT MAX(snapshot_version) FROM pep_dev.bronze.snapshot_version_control)

GROUP BY psr.entity_id

In [0]:
SELECT COUNT(*), COUNT(DISTINCT entity_id) FROM pep_dev.eval.er_records;

WITH groups  AS (
SELECT canonical_id, 
       count(*) as n_records 
 FROM pep_dev.eval.er_records 
GROUP BY canonical_id

)
SELECT  
    SUM(n_records)  filter(WHERE n_records>1)  
    as records_in_multi_groups,
    max(n_records) largest_group,
    count_if(n_records>1) multi_record_groups,
    count(*) all_groups,
    SUM(n_records*(n_records-1)/2) AS possible_answer_key_pairs

  FROM groups ;

In [0]:
SELECT DISTINCT(_last_seen_snapshot) FROM pep_dev.eval.er_records

In [0]:
SELECT count(*),count_if(size(names) = 0) FROM pep_dev.eval.er_records
WHERE size(inn_codes)>0
        OR
      size(ogrn_codes)>0  
      OR
      size(registration_numbers)>0
        OR
      size(tax_numbers)>0  
      OR
      size(lei_codes) > 0

In [0]:
SELECT * FROM pep_dev.eval.er_records

**PHASE 2**

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.er_names
AS
SELECT 
    entity_id,
    country,
    explode(names) as raw_name
FROM pep_dev.eval.er_records;

In [0]:
WITH v1_cleaned_table as (
SELECT 
lower(raw_name) as raw_name,
regexp_replace(lower(raw_name), '[““”«»„‘.,()-/&:;№#]',' ')
AS cleaned_string
FROM pep_dev.eval.er_names
WHERE raw_name LIKE '%«%'
)

SELECT regexp_replace(cleaned_string,'`' ,''), raw_name FROM v1_cleaned_table